# 실제 모델 학습에 사용되지 않은 파일입니다
- 시행착오

## 531 프로그램 학습 데이터 수집
- 위키피디아, 나무위키를 통한 데이터 수집
- AI 허브 - 초거대 AI 헬스케어 질의응답 데이터 수집(운동과 관련된 질병 데이터만 수집)
- ~~공동 모델 학습을 위한 데이터를 수집한다.~~
- ~~위키피디아, 나무위키에서 수집~~
- ~~[[AI 허브] 초거대 AI 헬스케어 질의응답 데이터](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=&topMenu=&aihubDataSe=data&dataSetSn=71762)~~
    ~~근골격질환/~~
    ~~관절염~~

    ~~염좌~~
    ~~요추추간판 탈출증~~
    ~~요추관 협착증~~

    ~~기타/~~
    ~~비만~~

    ~~순환기질환/~~
    ~~고혈압~~

    ~~저혈압~~
    ~~하지정맥류~~

In [1]:
import wikipedia

def fetch_wikipedia_text(query, lang="ko"):
    wikipedia.set_lang(lang)
    try:
        page = wikipedia.page(query)
        return page.content
    except Exception as e:
        print(f"❌ Error for {query}: {e}")
        return None

In [2]:

topics = ["근력 트레이닝", "파워리프팅", "웨이트 트레이닝", "근비대", "벤치 프레스", "데드리프트", "스쿼트"]
with open("./dataset/pre-training/wikipedia.txt", "w", encoding="utf-8") as f:
    for topic in topics:
        text = fetch_wikipedia_text(topic)
        if text:
            f.write(f"# {topic}\n{text}\n\n")
print("✅ 위키백과 텍스트 수집 완료")


✅ 위키백과 텍스트 수집 완료


In [7]:
import re

# 파일 로드
with open("./dataset/pre-training/wikipedia.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 1. 제목 구분자 제거 (#, 숫자. 등)
cleaned_text = re.sub(r"#\s*", "", raw_text)
cleaned_text = re.sub(r"\n\d+\.\s*", "\n", cleaned_text)

# 2. 괄호 안 각주, 링크 제거
cleaned_text = re.sub(r"\[.*?\]", "", cleaned_text)  # 각주나 링크
cleaned_text = re.sub(r"\(.*?(나무위키|https?:\/\/).*?\)", "", cleaned_text)  # 외부 링크 포함 괄호 제거

# 3. HTML 잔재 제거 (혹시 있을 경우)
cleaned_text = re.sub(r"<[^>]+>", "", cleaned_text)

# 4. 중복 개행 제거, 양쪽 공백 정리
cleaned_text = re.sub(r"\n{2,}", "\n", cleaned_text).strip()

# 5. 너무 짧거나 긴 문장은 제거 (5자 이하 또는 500자 이상)
sentences = cleaned_text.split("\n")
filtered_sentences = [s.strip() for s in sentences if 10 <= len(s.strip()) <= 500]

# 6. 문장 연결
final_text = "\n".join(filtered_sentences)

# 저장
output_path = "./dataset/pre-training/wikipedia_pretraining_cleaned.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_text)

output_path


'./dataset/pre-training/wikipedia_pretraining_cleaned.txt'

In [3]:
# namuwiki_scraper.py

# 필요한 라이브러리 설치
# pip install selenium beautifulsoup4

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from urllib.parse import quote
import time
import os

# 크롤링 대상 키워드 리스트
KEYWORDS = ["파워리프팅", "웨이트 트레이닝", "무산소 운동", "스트렝스 트레이닝", "벤치 프레스", "데드리프트", "스쿼트", "프레스"]

# 저장 경로
OUTPUT_PATH = "./dataset/pre-training/namuwiki.txt"

# 나무위키 본문 수집 함수
def fetch_namuwiki_content(keyword):
    encoded = quote(keyword)
    url = f"https://namu.wiki/w/{encoded}"

    # headless 크롬 브라우저 설정
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    driver = webdriver.Chrome(options=options)

    try:
        print(f"📄 크롤링 중: {keyword}")
        driver.get(url)
        time.sleep(2)  # JavaScript 렌더링 대기

        soup = BeautifulSoup(driver.page_source, "html.parser")
        content_div = soup.find("div", class_="n9-x2yjp")  # 본문 클래스

        if content_div:
            text = content_div.get_text(separator="\n").strip()
            return text
        else:
            print(f"⚠️ 본문을 찾을 수 없습니다: {url}")
            return None

    except Exception as e:
        print(f"❌ 예외 발생: {keyword} - {e}")
        return None

    finally:
        driver.quit()

# 텍스트 저장 함수
def save_to_file(topic, text, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"\n\n# {topic}\n{text}\n")

# 실행 메인 함수
def main():
    if os.path.exists(OUTPUT_PATH):
        os.remove(OUTPUT_PATH)

    for keyword in KEYWORDS:
        content = fetch_namuwiki_content(keyword)
        if content:
            save_to_file(keyword, content, OUTPUT_PATH)
        time.sleep(10)  # 요청 간 시간 간격 유지

    print(f"\n✅ 수집 완료. 결과 파일: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()


📄 크롤링 중: 파워리프팅
📄 크롤링 중: 웨이트 트레이닝
📄 크롤링 중: 무산소 운동
📄 크롤링 중: 스트렝스 트레이닝
📄 크롤링 중: 벤치 프레스
📄 크롤링 중: 데드리프트
📄 크롤링 중: 스쿼트
📄 크롤링 중: 프레스

✅ 수집 완료. 결과 파일: ./dataset/pre-training/namuwiki.txt


In [4]:
import re
import json

with open("./dataset/pre-training/namuwiki.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

sections = re.split(r'\n\d+\.\s+', raw_text)  # 대주제 기준 나누기
dataset = []

for section in sections:
    lines = section.strip().split('\n')
    content = " ".join(line.strip() for line in lines if line.strip())
    if len(content) < 30 or len(content) > 1000:
        continue

    question = f"{lines[0].strip()}에 대해 설명해줘." if lines else "다음 내용에 대해 설명해줘."
    dataset.append({
        "instruction": question,
        "input": "",
        "output": content
    })

# JSONL로 저장
with open("instruction_data.jsonl", "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


In [6]:
import re

# 파일 로드
with open("./dataset/pre-training/namuwiki.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 1. 제목 구분자 제거 (#, 숫자. 등)
cleaned_text = re.sub(r"#\s*", "", raw_text)
cleaned_text = re.sub(r"\n\d+\.\s*", "\n", cleaned_text)

# 2. 괄호 안 각주, 링크 제거
cleaned_text = re.sub(r"\[.*?\]", "", cleaned_text)  # 각주나 링크
cleaned_text = re.sub(r"\(.*?(나무위키|https?:\/\/).*?\)", "", cleaned_text)  # 외부 링크 포함 괄호 제거

# 3. HTML 잔재 제거 (혹시 있을 경우)
cleaned_text = re.sub(r"<[^>]+>", "", cleaned_text)

# 4. 중복 개행 제거, 양쪽 공백 정리
cleaned_text = re.sub(r"\n{2,}", "\n", cleaned_text).strip()

# 5. 너무 짧거나 긴 문장은 제거 (5자 이하 또는 500자 이상)
sentences = cleaned_text.split("\n")
filtered_sentences = [s.strip() for s in sentences if 10 <= len(s.strip()) <= 500]

# 6. 문장 연결
final_text = "\n".join(filtered_sentences)

# 저장
output_path = "./dataset/pre-training/namuwiki_pretraining_cleaned.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_text)

output_path


'./dataset/pre-training/namuwiki_pretraining_cleaned.txt'